# 🚀 CTDDG — Full Pipeline RunnerRuns the **entire** CTDDG pipeline end-to-end in a single notebook.Use this for unattended batch execution on the cluster.**Estimated time:** 12-24 hours on 2× L40 GPUs

In [ ]:
import os, sys, time, subprocessCTDDG_ROOT = os.environ.get("CTDDG_ROOT", os.path.dirname(os.getcwd()))os.environ["CTDDG_ROOT"] = CTDDG_ROOTos.environ["MXNET_CUDNN_LIB_CHECKING"] = "0"os.chdir(CTDDG_ROOT)import mxnet as mxprint(f"Root: {CTDDG_ROOT}")print(f"GPUs: {mx.context.num_gpus()}")!nvidia-smi --query-gpu=index,name --format=csv,noheader

## Pipeline Configuration

In [ ]:
# What to run (set False to skip a stage)RUN_PRETRAINING = TrueRUN_FINETUNING = TrueRUN_GENERATION = TrueRUN_EVALUATION = TrueRUN_DOCKING = False  # Often done separatelyDATASET_INDEX = 1N_SAMPLES = 1000RUN_ID = 1

## Step 0: Setup

In [ ]:
# Patch paths and convert ChEMBL!python scripts/patch_paths.py {CTDDG_ROOT}chembl_src = os.path.join(CTDDG_ROOT, "data", "chembl", "chembl.txt")chembl_dst = os.path.join(CTDDG_ROOT, "data", "chembl", "chembl_final.txt")if os.path.exists(chembl_src) and not os.path.exists(chembl_dst):    !python scripts/convert_chembl_format.py {chembl_src} {chembl_dst}for d in ["outputs/pretrain/logs", "outputs/docking"]:    os.makedirs(os.path.join(CTDDG_ROOT, d), exist_ok=True)print("✅ Setup complete")

## Execute Pipeline Stages

In [ ]:
def run_notebook(nb_path, label, timeout=172800):    print(f"\n{'='*60}")    print(f"  ▶ {label}")    print(f"{'='*60}")    t0 = time.time()    name = os.path.splitext(os.path.basename(nb_path))[0]    result = subprocess.run(        [sys.executable, "-m", "jupyter", "nbconvert",         "--to", "notebook", "--execute",         f"--ExecutePreprocessor.timeout={timeout}",         "--ExecutePreprocessor.kernel_name=ctddg_env",         f"--output={name}_executed.ipynb",         nb_path],        capture_output=True, text=True, cwd=CTDDG_ROOT    )    elapsed = time.time() - t0    if result.returncode == 0:        print(f"  ✅ {label} completed in {elapsed/60:.1f} min")        return True    else:        print(f"  ❌ {label} FAILED after {elapsed/60:.1f} min")        print(f"  Error: {result.stderr[-1000:]}")        return Falsepipeline_start = time.time()if RUN_PRETRAINING:    ok = run_notebook("code/pretraining.ipynb", "Stage 1: Pretraining")    if not ok: raise RuntimeError("Pretraining failed")if RUN_FINETUNING:    # Fine-tuning needs special handling since finetune_cell.py    # depends on pretraining.ipynb classes    print("\n" + "="*60)    print("  ▶ Stage 2: Fine-Tuning")    print("="*60)    print("  ℹ️  Run notebook 02_finetuning.ipynb for this stage")    print("  (Fine-tuning requires pretraining class definitions in-memory)")if RUN_GENERATION:    run_notebook("code/Generating_samples.ipynb", "Stage 3: Generation")if RUN_EVALUATION:    run_notebook("code/Evaluation_metrics.ipynb", "Stage 4: Evaluation")if RUN_DOCKING:    run_notebook("code/Molecular_docking.ipynb", "Stage 5: Docking")total = time.time() - pipeline_startprint(f"\n{'='*60}")print(f"  🎉 Pipeline completed in {total/3600:.1f} hours")print(f"{'='*60}")